In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
#import fiona
import mercantile #to convert quadkey to geometry
from shapely.geometry import box
import time

import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)
config
#import fiona
#fiona.listlayers("ADMIN_EXPRESS.gpkg")


{'input_data': {'file1': '../data/raw/2024-10-01_performance_mobile_tiles.parquet',
  'file2': '../data/raw/ADE_4-0_GPKG_WGS84G_FRA-ED2026-01-19.gpkg'},
 'output_data': {'file1': '../data/clean/file1_cleaned.csv',
  'file2': '../data/clean/file2_cleaned.csv'}}

In [5]:
#REd the parquet file
internet_perf_df = pd.read_parquet(config['input_data']['file1'])
internet_perf_df = internet_perf_df.drop(columns = ['tile', 'tile_x', 'tile_y'])

In [6]:
#Extract regions, depts and communes from GPKG file
regions = gpd.read_file(config['input_data']['file2'], layer = 'REGION')

depts = gpd.read_file(config['input_data']['file2'], layer = 'DEPARTEMENT')

communes = gpd.read_file(config['input_data']['file2'], layer = 'COMMUNE')


In [ ]:
departments.plot()

In [9]:
# Create a function to convert quadkeys to geometry
def quadkey_to_geom(qk):
    tile = mercantile.quadkey_to_tile(qk)
    bounds = mercantile.bounds(tile)
    
    return box(
        bounds.west,
        bounds.south,
        bounds.east,
        bounds.north
    )

#Apply function of my geopadas df: quakey to geometry and add geometry column
start_time = time.time()
print("Converting quadkeys to geometry in progress...")
internet_perf_df["geometry"] = internet_perf_df["quadkey"].apply(quadkey_to_geom)
end_time = time.time()
print(f"Converting quadkeys to geometry finished. Time taken: {end_time - start_time: .2f}seconds")

Converting quadkeys to geometry in progress...


Converting quadkeys to geometry finished. Time taken:  250.24seconds


In [10]:
#Convert pandas df to GeoDataFrame
def convert_df_to_gdf(df):
    """
    Input: pandas df
    convert pandas to geopandas df
    Outpu: geopandas df
    """
    start_time = time.time()
    print("Converting Pandas DF to GeoPandas DF in progress ...")
    df1 = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326") #EPSG:4326 = WGS84     
    end_time = time.time()
    print(f"Pandas df is converted to geopandas df. Time take is: {end_time - start_time: .2f}seconds")

    return df1

internet_perf_gdf = convert_df_to_gdf(internet_perf_df)
internet_perf_gdf.head()

Converting Pandas DF to GeoPandas DF in progress ...


Pandas df is converted to geopandas df. Time take is:  0.30seconds


,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices,geometry
0,0022310222030300,16997,2239,744,1542.0,737.0,1,1,"POLYGON ((-163.00964 69.73904, -163.00964 69.7..."
1,0022332203013331,4839,948,43,91.0,88.0,1,1,"POLYGON ((-162.59766 66.89775, -162.59766 66.8..."
2,0022332203031110,169938,23880,45,516.0,439.0,3,2,"POLYGON ((-162.60315 66.89344, -162.60315 66.8..."
3,0022332203031113,10550,15278,209,1025.0,1112.0,1,1,"POLYGON ((-162.59766 66.89128, -162.59766 66.8..."
4,0022332203102232,4722,912,42,73.0,75.0,1,1,"POLYGON ((-162.58118 66.8956, -162.58118 66.89..."


In [37]:
# make sjoin for departments and regions and communes
def join_admin_to_perf_gdf(gdf1, gdf2):
    """
    Input: geopandas df (performance gdf & admin gdf)
    join geopandas df to geo to geopandas df
    Outpu: geopandas df combined
    """
    gdf0 = gpd.sjoin(
        gdf1,
        gdf2,
        how = "left",
        predicate = "within"
    )
    gdf0 = gdf0.drop(columns = "index_right")
    return gdf0

internet_perf_reg_gdf =  join_admin_to_perf_gdf(internet_perf_gdf, regions)
internet_perf_dept_gdf =  join_admin_to_perf_gdf(internet_perf_reg_gdf, depts)
internet_perf_admin_gdf =  join_admin_to_perf_gdf(internet_perf_dept_gdf, communes)

#generate the gdf with all admin hierarchy; region/dpt/commune
internet_perf_admin_gdf

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices,geometry,cleabs_left,...,date_du_recensement,organisme_recenseur,code_insee_du_canton,code_insee_de_l_arrondissement,code_insee_du_departement,code_insee_de_la_region_right,codes_siren_des_epci,code_siren,code_postal,superficie_cadastrale
0,0022310222030300,16997,2239,744,1542.0,737.0,1,1,"POLYGON ((-163.00964 69.73904, -163.00964 69.7...",NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0022332203013331,4839,948,43,91.0,88.0,1,1,"POLYGON ((-162.59766 66.89775, -162.59766 66.8...",NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0022332203031110,169938,23880,45,516.0,439.0,3,2,"POLYGON ((-162.60315 66.89344, -162.60315 66.8...",NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0022332203031113,10550,15278,209,1025.0,1112.0,1,1,"POLYGON ((-162.59766 66.89128, -162.59766 66.8...",NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0022332203102232,4722,912,42,73.0,75.0,1,1,"POLYGON ((-162.58118 66.8956, -162.58118 66.89...",NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3551262,3131120220100031,98579,54717,22,2447.0,678.0,1,1,"POLYGON ((168.94775 -46.57019, 168.94775 -46.5...",NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3551263,3131120221020130,54913,21178,50,791.0,1194.0,1,1,"POLYGON ((169.14001 -46.63058, 169.14001 -46.6...",NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3551264,3131120230000011,8179,1131,313,1275.0,1093.0,1,1,"POLYGON ((169.4751 -46.56264, 169.4751 -46.558...",NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3551265,3131120230000032,45462,10434,50,839.0,639.0,1,1,"POLYGON ((169.4696 -46.57397, 169.4696 -46.570...",NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
# Renaming and selecting columns
internet_perf_admin_gdf.columns
selected_col = ['avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', 'avg_lat_down_ms',
               'avg_lat_up_ms', 'tests', 'devices','nom_officiel_en_majuscules_left',
               'code_insee_left', 'code_siren_left','nom_officiel_en_majuscules_right',
                'code_insee_right', 'nom_officiel_en_majuscules', 'statut',
               'code_insee', 'population', 'date_du_recensement',
               'code_postal', 'superficie_cadastrale'] #'quadkey',  'geometry',  

new_col_names = ['avg_down_mbps', 'avg_up_mbps', 'avg_lat_ms', 'avg_lat_down_ms',
                'avg_lat_up_ms', 'nbr_tests', 'nbr_devices','region_name',
               'code_insee_region', 'code_siren_region','department_name',
                'code_insee_dept', 'commune_name', 'commune_status',
                'code_insee_comm', 'comm_population', 'date_du_recensement',
                'zip_code', 'superficie_cadastrale'
                ]

internet_perf_admin_gdf_red_col = internet_perf_admin_gdf[selected_col]

internet_perf_admin_gdf_red_col.columns = new_col_names
internet_perf_admin_gdf_fr = internet_perf_admin_gdf_red_col.dropna(subset = ['department_name'])#.reset_index()

internet_perf_admin_gdf_fr.head()
internet_perf_admin_gdf_fr.isna().sum()

avg_down_mbps                0
avg_up_mbps                  0
avg_lat_ms                   0
avg_lat_down_ms           1482
avg_lat_up_ms              544
nbr_tests                    0
nbr_devices                  0
region_name                  0
code_insee_region            0
code_siren_region            0
department_name              0
code_insee_dept              0
commune_name             13947
commune_status           13947
code_insee_comm          13947
comm_population          13947
date_du_recensement      13947
zip_code                 15051
superficie_cadastrale    13947
dtype: int64

In [41]:
internet_perf_admin_gdf_fr.info()

<class 'pandas.DataFrame'>
Index: 68022 entries, 512007 to 3361040
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   avg_down_mbps          68022 non-null  int64         
 1   avg_up_mbps            68022 non-null  int64         
 2   avg_lat_ms             68022 non-null  int64         
 3   avg_lat_down_ms        66529 non-null  float64       
 4   avg_lat_up_ms          67468 non-null  float64       
 5   nbr_tests              68022 non-null  int64         
 6   nbr_devices            68022 non-null  int64         
 7   region_name            68022 non-null  str           
 8   code_insee_region      68022 non-null  str           
 9   code_siren_region      68022 non-null  str           
 10  department_name        67118 non-null  str           
 11  code_insee_dept        67118 non-null  str           
 12  code_insee_region2     67118 non-null  str           
 13  commune_na

In [ ]:

ookla_dept = result.dropna(subset = ['nom_officiel_en_majuscules'])
ookla_dept.nom_officiel_en_majuscules.unique()
result.shape #result.shape = (3551267, 19)
ookla_dept.shape #(67118, 19)
